## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


## 2. Load and Explore Data


In [ ]:
# Detect environment and set data paths
import os

# Check if running on Kaggle
if os.path.exists('/kaggle/input'):
    # Kaggle environment - dataset: "heartbeat" (ECG Heartbeat Categorization Dataset)
    DATA_PATH = '/kaggle/input/heartbeat'
    OUTPUT_PATH = '/kaggle/working'
    print("Running on Kaggle")
else:
    # Local environment
    DATA_PATH = 'data'
    OUTPUT_PATH = '.'
    print("Running locally")

# Load the MIT-BIH dataset
train_df = pd.read_csv(f'{DATA_PATH}/mitbih_train.csv', header=None)
test_df = pd.read_csv(f'{DATA_PATH}/mitbih_test.csv', header=None)

print(f"\nTraining data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")


In [ ]:
# Display first few rows
print("First 5 rows of training data:")
train_df.head()


In [ ]:
# The last column (187) contains the class labels
# Columns 0-186 contain the ECG signal features

# Separate features and labels
X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values

X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


## 3. Data Visualization


In [ ]:
# Class distribution
class_names = ['Normal (N)', 'Supraventricular (S)', 'Ventricular (V)', 'Fusion (F)', 'Unknown (Q)']

plt.figure(figsize=(10, 6))
unique, counts = np.unique(y_train, return_counts=True)
plt.bar([class_names[int(i)] for i in unique], counts, color=['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6'])
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class Distribution in Training Data')
plt.xticks(rotation=45)
for i, (u, c) in enumerate(zip(unique, counts)):
    plt.text(i, c + 500, str(c), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nClass distribution:")
for u, c in zip(unique, counts):
    print(f"  Class {int(u)} ({class_names[int(u)]}): {c} samples ({c/len(y_train)*100:.2f}%)")


In [ ]:
# Visualize sample ECG signals for each class
fig, axes = plt.subplots(5, 1, figsize=(14, 12))

for i, ax in enumerate(axes):
    # Get a sample from each class
    idx = np.where(y_train == i)[0][0]
    ax.plot(X_train[idx], color=['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6'][i], linewidth=1.5)
    ax.set_title(f'Class {i}: {class_names[i]}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time Steps')
    ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Sample ECG Heartbeats for Each Class', fontsize=14, fontweight='bold', y=1.02)
plt.show()


## 4. Data Preprocessing


In [ ]:
# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data standardization complete!")
print(f"X_train_scaled mean: {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std: {X_train_scaled.std():.6f}")


In [ ]:
# Convert labels to integers
y_train = y_train.astype(int)
y_test = y_test.astype(int)

print(f"Unique classes in training: {np.unique(y_train)}")
print(f"Unique classes in test: {np.unique(y_test)}")


## 5. Build and Train the Classification Model


In [ ]:
# Initialize Random Forest Classifier
# Using Random Forest as it handles imbalanced data well and provides feature importance

rf_model = RandomForestClassifier(
    n_estimators=100,      # Number of trees
    max_depth=20,          # Maximum depth of trees
    min_samples_split=5,   # Minimum samples to split a node
    min_samples_leaf=2,    # Minimum samples in leaf node
    random_state=42,       # For reproducibility
    n_jobs=-1,             # Use all CPU cores
    class_weight='balanced' # Handle class imbalance
)

print("Model initialized!")
print(rf_model)


In [ ]:
# Train the model
print("Training the model... This may take a few minutes.")

import time
start_time = time.time()

rf_model.fit(X_train_scaled, y_train)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.2f} seconds!")


## 6. Model Evaluation


In [ ]:
# Make predictions
y_pred_train = rf_model.predict(X_train_scaled)
y_pred_test = rf_model.predict(X_test_scaled)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


In [ ]:
# Classification Report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred_test, target_names=class_names))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Normalized Confusion Matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='RdYlGn', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix (Recall per Class)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Feature Importance Analysis


In [ ]:
# Get feature importances
feature_importance = rf_model.feature_importances_

# Plot top 30 most important features
top_n = 30
indices = np.argsort(feature_importance)[-top_n:][::-1]

plt.figure(figsize=(12, 8))
plt.bar(range(top_n), feature_importance[indices], color='steelblue')
plt.xlabel('Feature Index (Time Step)', fontsize=12)
plt.ylabel('Importance', fontsize=12)
plt.title(f'Top {top_n} Most Important Features', fontsize=14, fontweight='bold')
plt.xticks(range(top_n), indices, rotation=45)
plt.tight_layout()
plt.show()

print(f"\nTop 10 most important time steps: {indices[:10]}")


In [ ]:
# Visualize feature importance on ECG signal
plt.figure(figsize=(14, 6))

# Plot a sample ECG
sample_ecg = X_train[0]
plt.plot(sample_ecg, 'b-', alpha=0.7, label='ECG Signal')

# Highlight important features
importance_scaled = (feature_importance - feature_importance.min()) / (feature_importance.max() - feature_importance.min())
for i, imp in enumerate(importance_scaled):
    if imp > 0.5:  # Highlight features with importance > 50%
        plt.axvline(x=i, color='red', alpha=imp*0.5, linewidth=1)

plt.xlabel('Time Steps', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.title('ECG Signal with Important Features Highlighted (Red)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Model Summary and Conclusions


In [ ]:
# Summary
print("="*60)
print("MODEL SUMMARY")
print("="*60)
print(f"\nModel: Random Forest Classifier")
print(f"Number of Trees: 100")
print(f"Max Depth: 20")
print(f"\nDataset:")
print(f"  - Training samples: {len(y_train):,}")
print(f"  - Test samples: {len(y_test):,}")
print(f"  - Features: {X_train.shape[1]}")
print(f"  - Classes: {len(class_names)}")
print(f"\nPerformance:")
print(f"  - Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"  - Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"  - Training Time: {training_time:.2f} seconds")
print("\n" + "="*60)


In [ ]:
# Per-class metrics
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)

print("\nPer-Class Metrics:")
print("-" * 70)
print(f"{'Class':<25} {'Precision':>12} {'Recall':>12} {'F1-Score':>12} {'Support':>10}")
print("-" * 70)
for i, name in enumerate(class_names):
    print(f"{name:<25} {precision[i]:>12.4f} {recall[i]:>12.4f} {f1[i]:>12.4f} {support[i]:>10}")
print("-" * 70)


## 9. Save the Model (Optional)


In [ ]:
# Save the trained model
import joblib

# Save model and scaler (uses OUTPUT_PATH defined earlier)
joblib.dump(rf_model, f'{OUTPUT_PATH}/ecg_classifier_model.joblib')
joblib.dump(scaler, f'{OUTPUT_PATH}/ecg_scaler.joblib')

print(f"Model saved as '{OUTPUT_PATH}/ecg_classifier_model.joblib'")
print(f"Scaler saved as '{OUTPUT_PATH}/ecg_scaler.joblib'")


In [ ]:
# Example: Load and use the saved model
# loaded_model = joblib.load(f'{OUTPUT_PATH}/ecg_classifier_model.joblib')
# loaded_scaler = joblib.load(f'{OUTPUT_PATH}/ecg_scaler.joblib')

# # Make prediction on new data
# new_sample = X_test[0].reshape(1, -1)
# new_sample_scaled = loaded_scaler.transform(new_sample)
# prediction = loaded_model.predict(new_sample_scaled)
# print(f"Predicted class: {class_names[prediction[0]]}")


---
## Conclusions

In this practical, we built a **Random Forest Classifier** to classify ECG heartbeats into 5 categories:

1. **Normal (N)** - Normal heartbeat
2. **Supraventricular (S)** - Supraventricular ectopic beat
3. **Ventricular (V)** - Ventricular ectopic beat
4. **Fusion (F)** - Fusion beat
5. **Unknown (Q)** - Unknown beat

### Key Findings:
- The model achieves good accuracy on the test set
- Class imbalance is handled using `class_weight='balanced'`
- Feature importance analysis reveals which time steps in the ECG signal are most discriminative
- The model can be saved and loaded for future predictions

### Potential Improvements:
- Try other algorithms (XGBoost, SVM, Neural Networks)
- Apply data augmentation techniques
- Use oversampling (SMOTE) for minority classes
- Perform hyperparameter tuning with GridSearchCV
